<a href="https://colab.research.google.com/github/ahersakshi/Major_Project/blob/main/Major.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import re
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import LSTM, Dense
import urllib.request

# 1. Load and clean text (you can use your custom text or a URL for a corpus)
url = "https://raw.githubusercontent.com/dwyl/english-words/master/words.txt"
response = urllib.request.urlopen(url)
text = response.read().decode('utf-8').lower()

# Clean the text
text = re.sub(r'[^a-zA-Z .,!?\'\"]+', ' ', text)

# 2. Create character to integer mapping
chars = sorted(list(set(text)))
char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for c, i in char_to_int.items()}

# 3. Create sequences
seq_length = 40
step = 1
sequences = []
next_chars = []

for i in range(0, len(text) - seq_length, step):
    sequences.append(text[i:i + seq_length])
    next_chars.append(text[i + seq_length])

print("Total sequences:", len(sequences))

# 4. Encode sequences using integer encoding (no one-hot encoding)
X = np.zeros((len(sequences), seq_length), dtype=np.int32)  # Integer encoding
y = np.zeros((len(sequences),), dtype=np.int32)  # Integer encoding for output

for i, seq in enumerate(sequences):
    for t, char in enumerate(seq):
        X[i, t] = char_to_int[char]  # Integer encoding for X
    y[i] = char_to_int[next_chars[i]]  # Integer encoding for y

# 5. Build LSTM model (use softmax for multi-class classification)
model = Sequential()
model.add(LSTM(128, input_shape=(seq_length,)))
model.add(Dense(len(chars), activation='softmax'))  # Output layer size is equal to number of characters
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 6. Train the model
history = model.fit(X, y, batch_size=8, epochs=100, verbose=1)

# 7. Plot training loss and accuracy
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Loss')
plt.plot(history.history['accuracy'], label='Accuracy')
plt.title('Training Loss & Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

# 8. Text generation function using temperature
def generate_text(seed_text, length=200, temperature=1.0):
    generated = seed_text.lower()
    for _ in range(length):
        x_pred = np.zeros((1, seq_length), dtype=np.int32)  # Integer encoding for input
        for t, char in enumerate(seed_text):
            if char in char_to_int:
                x_pred[0, t] = char_to_int[char]
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = np.argmax(sample(preds, temperature))
        next_char = int_to_char[next_index]
        generated += next_char
        seed_text = seed_text[1:] + next_char
    return generated

# 9. Example text generation
seed_text = "to be, or not to be, that is the que"
print("\nGenerated Text:\n")
print(generate_text(seed_text, length=500, temperature=0.8))


Total sequences: 4860979


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ValueError: Input 0 of layer "lstm" is incompatible with the layer: expected ndim=3, found ndim=2. Full shape received: (None, 40)